In [ ]:
import polars as pl
from pathlib import Path
import jellyfish
from nltk.corpus import names
import nltk
#nltk.download('names')
import re

[nltk_data] Downloading package names to
[nltk_data]     C:\Users\ganon\AppData\Roaming\nltk_data...
[nltk_data]   Package names is already up-to-date!


In [127]:
samples_df = pl.read_parquet("jails_pdfs_samples.parquet")

In [128]:
samples_df

"Deceased Cause, Date, and Time",Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Name,Facility Type,Phone Number,Address,Date,Time of Day,AM or PM,Occurrence,Table Contents,Injuries?,Resulting Death?,Report ID,OCR_Confidence,OCR_Word_Count,OCR_Text_Detected,Cook County?
str,str,str,str,str,str,str,str,str,str,str,str,list[str],list[str],str,str,str,f64,i64,bool,str
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""-acility Name: Williamson Coun…","""County ""","""__ Telephone #: 618-998-2100 ""","""Address: 404 N. Van Buren Mari…","""Date of Occurrence: 07/22/2022…","""_ Time of Occurrence: 2:3 ""","""No filled box found""","[""(J Other (specify): "", ""Restraints Used "", … ""Offense oO ""]","[""Abels, Kristin"", ""02/21/1985"", ""Poss Cannabis""]","""No injuries""","""No filled box found""","""UO - JDSU (July 2022)_p3""",86.5,280,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: STEPHENSON COUN…","""County ""","""____—« Telephone #: 815-235-82…","""\ddress: 1680 E. SINGER DRIVE …","""Date of Occurrence: 07/03/2022…","""_ Time of Occurrence: 1038 ""","""No filled box found""","[""ther (specify): ERC "", ""Suicide Allempl (method) Hanging ""]","[""dams, Rayvonna"", "": 12/14/2002"", … ""seetptess""]","""No injuries""","""No ""","""UO - JDSU (July 2022)_p4""",88.392617,298,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Cook County Dep…","""Error: less than two boxes fou…","""Telephone #: 773.674.3474 O IL…","""Aqaress: SIUM Y4- SGV MIVioIVt…","""Date of Occurrence: 7/26/2022 …","""SSPE WE Nw EE Pe Ne ew [A Giii…","""Check time, no AM or PM for Co…","[""1er: "", "", Spray Used "", … ""Suicide Attempt (method) ""]","[""Michael A Aguirre"", ""12/13/1994"", ""10/6/2021""]","""No injuries""","""Error: less than two boxes fou…","""UO - JDSU (July 2022)_p5""",88.283154,279,true,"""Cook County"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Cook County Dep…","""Error: less than two boxes fou…","""Telephone #: 773.674.3474 @) I…","""Aqaress: SIUM Y4- SGM MIVioIVt…","""Date of Occurrence: 7/13/2022 …","""SSN WE EE Pe il i il T1 Giirci…","""Check time, no AM or PM for Co…","[""er: "", ""Fighting amon "", … ""ense T] ""]","[""Devin Aldridge"", ""9/21/1990"", … ""12/9/2021""]","""No injuries""","""No filled box found""","""UO - JDSU (July 2022)_p6""",88.024735,283,true,"""Cook County"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Cook County Dep…","""Error: less than two boxes fou…","""Telephone #: 773.674.3474 @) I…","""Aqaress: SIUM YS SGV MiVvioiVi…","""Date of Occurrence: 7/16/2022 …","""SSN WE EE Pe a ee I eC T1 Giic…","""Check time, no AM or PM for Co…","[""er: "", ""Fighting amon "", … ""ense T] ""]","[""Nathaniel T Alexander"", ""1/17/1966"", … ""7/10/2020""]","""No injuries""","""No filled box found""","""UO - JDSU (July 2022)_p7""",87.674825,286,true,"""Cook County"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Sangamon County…","""County ""","""\ 2 ae __- Telephone #: 217-75…","""\ddress: #1 Sheriff's Plaza Sp…","""Date of Occurrence: 07/05/2022…","""_ Time of Occurrence: 2315 ""","""on ""","[""Other (specify): hosp. transp ""]","[""| pe eee ——»"", ""a me a 2a eg, Sa"", … ""Sn ar""]","""No injuries""","""No filled box found""","""UO - JDSU (July 2022)_p392""",86.386719,256,true,null
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""‘acility Name: Cook County Dep…","""Count ""","""Telephone #: 773.674.3474 @) I…","""Aqaress: that = hed ih oOUvU0U…","""Date of Occurrence: 7/1/2022 T…","""SST WE Nw EE Eee aN eS Tl) Gii…","""Check time, no AM or PM for Co…","[""ver: "", ""> Spray Used "", … ""ense T] ""]","[""Raven A Wright"", ""10/12/1996"", ""7/1/2022""]","""No injuries""","""Error: less than two boxes fou…","""UO - JDSU (July 2022)_p393""",88.42807,285,true,"""Cook County"""
"""N/A""","""N/A""",

In [129]:
yes_count = (samples_df['Resulting Death?'] == 'Yes').sum()


print (yes_count)

0


In [130]:
entries = ["Restraints Used\n", "Smattery"]


def clean_occurrences(entries):

    actual_occurrences = ["Suicide (method)", "Suicide (attempt)", "Homicide", "Homicide Attempt", "Escape", "Escape Attempt",
                "Fire", "Serious Injury", "Battery", "Riot of Rebellion", "Sex Offense", "Assault on Staff",
                "Assault among Detainees", "Fighting among Detainees", "Restraints Used", "OC Spray Used", "Other (specify)"]
    
    if entries.is_empty():
        return ["Error, no occurrence found"]

    cleaned_list = []
    for term in entries:
        best_choice = None
        highest_jaro = 0.0
        for compare_term in actual_occurrences:
            jaro_score = jellyfish.jaro_similarity(term, compare_term)
            if jaro_score > highest_jaro:
                highest_jaro = jaro_score
                best_choice = compare_term

        if best_choice in ["Suicide (method)", "Suicide (attempt)", "Other (specify)"]:
            # Keep the original term (possibly has colon text, like "Other (specify): Fight")
            cleaned_list.append(term)
        else:
            if best_choice not in cleaned_list:
                cleaned_list.append(best_choice)

    return cleaned_list

In [ ]:
all_names = set(names.words())

test_list = ["Deaunte Campbell","3/12/2002","10/13/2021"]

def get_inmate_names(table_list):
    inmate_list = []
    pattern_punctuation = r"[^a-zA-Z0-9/ ]"
    for item in table_list:
        cleaned_item = re.sub(pattern_punctuation, '', item)
        tokens = cleaned_item.strip().split()
        for token in tokens:
            token = token.capitalize()
            if token in all_names:
                inmate_list.append(cleaned_item)
                break
    return inmate_list

get_inmate_names(test_list)

False


[]

In [132]:
samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_14132\3155367749.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))


In [133]:
samples_df = samples_df.with_columns(pl.col("Table Contents").map_elements(get_inmate_names).alias("Inmate Names"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_14132\3131983131.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col("Table Contents").map_elements(get_inmate_names).alias("Inmate Names"))


In [134]:
# Clean facility name:
def clean_facility_name(df):
    """Clean facility names using Polars expressions"""
    return df.with_columns(
        pl.col('Facility Name')
        # First, clean OCR artifacts
        .str.replace_all(r'\\n|\\\\|\\/', ' ') 
        .str.replace_all(r'\n', ' ')
        .str.replace_all(r'[\\\/]+', ' ')
        .str.replace_all(r'\s+', ' ') 
        .str.strip_chars()
        # Extract facility name after any variation of "Name:"
        .str.extract(r'cility Name:\s*(.+)', 1)
        # Additional cleanup of extracted names
        .str.replace_all(r'[^a-zA-Z\s\'-]', ' ')  # Keep only letters, spaces, apostrophes, hyphens
        .str.replace_all(r'\s+', ' ')  # Clean multiple spaces again
        .str.strip_chars()  # Final trim
        # Stop at any word ending in "ddres" and remove that word + everything after
        .str.replace_all(r'\s+(ddress|ddrace).*$', '')
        .str.replace_all(r"(?i)\bSheriff's\b", "Sheriffs")
        .str.to_titlecase()
        .alias('Cleaned Facility Name')
    )

samples_df = clean_facility_name(samples_df)

# Date cleanning
def clean_date_occurrence(df):
    return df.with_columns(
        pl.col("Date")
          # OCR cleanup (keep forward slashes)
          .str.replace_all(r"\\n|\\\\", " ")
          .str.replace_all(r"\n", " ")
          .str.replace_all(r"\s+", " ")
          .str.strip_chars()
          # Extract the MM/DD/YYYY text
          .str.extract(r"Date of Occurrence:\s*([0-9]{1,2}/[0-9]{1,2}/(20[0-9]{2}))", 1)
          # Parse & re‐format to zero‐padded MM/DD/YYYY
          .str.strptime(pl.Date, "%m/%d/%Y", strict=False)
          .alias("temp_date")
    ).with_columns(
        # Only keep dates between 2000 and 2025, otherwise set to null
        pl.when(pl.col("temp_date").dt.year().is_between(2000, 2025))
        .then(pl.col("temp_date").dt.strftime("%m/%d/%Y"))
        .otherwise(None)
        .alias("Cleaned Date")
    ).drop("temp_date")

samples_df = clean_date_occurrence(samples_df)

In [ ]:
result = (
    samples_df.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(10)
)

result

with pl.Config(
    tbl_formatting="MARKDOWN",
    tbl_hide_column_data_types=True,
    tbl_hide_dataframe_shape=True,):
    print(result)

Cleaned Occurrences,count
str,u32
"""Serious Injury,Fighting among …",93
"""Serious Injury,OC Spray Used,B…",71
"""Restraints Used""",26
"""Serious Injury,OC Spray Used,F…",14
"""Fighting among Detainees""",12
"""Error, no occurrence found""",11
"""Serious Injury,Fighting among …",8
"""OC Spray Used""",8
"""ther (specify): Hospital """,4


In [ ]:
result

In [136]:
samples_df = samples_df.drop(["OCR_Confidence", "OCR_Word_Count", "OCR_Text_Detected", "Occurrence"])

In [137]:
samples_df = samples_df.drop(["Facility Name", "Date"])

In [138]:
samples_df = samples_df.drop(["Cook County?"])

In [139]:
samples_df = samples_df.with_columns(pl.col("Cleaned Occurrences").list.join(",").alias("Cleaned Occurrences"))

samples_df = samples_df.with_columns(pl.col("Inmate Names").list.join(",").alias("Inmate Names"))

samples_df = samples_df.with_columns(pl.col("Table Contents").list.join(",").alias("Table Contents"))

In [141]:
samples_df = samples_df.select(["Report ID", "Facility Type", "Cleaned Facility Name", "Address", "Phone Number", "Cleaned Date", "Time of Day", "AM or PM", "Table Contents", "Inmate Names", "Cleaned Occurrences", "Injuries?", "Resulting Death?", 'Deceased Cause, Date, and Time', 'Deceased on Suicide Watch', 'Deceased Reporter', 'Deceased Examined by Physician', 'Deceased Signs of Illness'])

In [142]:
samples_df

Report ID,Facility Type,Cleaned Facility Name,Address,Phone Number,Cleaned Date,Time of Day,AM or PM,Table Contents,Inmate Names,Cleaned Occurrences,Injuries?,Resulting Death?,"Deceased Cause, Date, and Time",Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""UO - JDSU (July 2022)_p3""","""County ""","""Williamson County Jail""","""Address: 404 N. Van Buren Mari…","""__ Telephone #: 618-998-2100 ""","""07/22/2022""","""_ Time of Occurrence: 2:3 ""","""No filled box found""","""Abels, Kristin,02/21/1985,Poss…","""Abels Kristin""","""(J Other (specify): ,Restraint…","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""UO - JDSU (July 2022)_p4""","""County ""","""Stephenson County Jail""","""\ddress: 1680 E. SINGER DRIVE …","""____—« Telephone #: 815-235-82…","""07/03/2022""","""_ Time of Occurrence: 1038 ""","""No filled box found""","""dams, Rayvonna,: 12/14/2002,: …","""""","""ther (specify): ERC ,Suicide A…","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""UO - JDSU (July 2022)_p5""","""Error: less than two boxes fou…","""Cook County Department Of Corr…","""Aqaress: SIUM Y4- SGV MIVioIVt…","""Telephone #: 773.674.3474 O IL…","""07/26/2022""","""SSPE WE Nw EE Pe Ne ew [A Giii…","""Check time, no AM or PM for Co…","""Michael A Aguirre,12/13/1994,1…","""Michael A Aguirre""","""Serious Injury,OC Spray Used,S…","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""UO - JDSU (July 2022)_p6""","""Error: less than two boxes fou…","""Cook County Department Of Corr…","""Aqaress: SIUM Y4- SGM MIVioIVt…","""Telephone #: 773.674.3474 @) I…","""07/13/2022""","""SSN WE EE Pe il i il T1 Giirci…","""Check time, no AM or PM for Co…","""Devin Aldridge,9/21/1990,8/9/2…","""Devin Aldridge,Alphonso W Joyn…","""Serious Injury,Fighting among …","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""UO - JDSU (July 2022)_p7""","""Error: less than two boxes fou…","""Cook County Department Of Corr…","""Aqaress: SIUM YS SGV MiVvioiVi…","""Telephone #: 773.674.3474 @) I…","""07/16/2022""","""SSN WE EE Pe a ee I eC T1 Giic…","""Check time, no AM or PM for Co…","""Nathaniel T Alexander,1/17/196…","""Nathaniel T Alexander,Michael …","""Serious Injury,Fighting among …","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""UO - JDSU (July 2022)_p392""","""County ""","""Sangamon County Adult Detentio…","""\ddress: #1 Sheriff's Plaza Sp…","""\ 2 ae __- Telephone #: 217-75…","""07/05/2022""","""_ Time of Occurrence: 2315 ""","""on ""","""| pe eee ——»,a me a 2a eg, Sa,…","""""","""Other (specify): hosp. transp ""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""UO - JDSU (July 2022)_p393""","""Count ""","""Cook County Department Of Corr…","""Aqaress: that = hed ih oOUvU0U…","""Telephone #: 773.674.3474 @) I…","""07/01/2022""","""SST WE Nw EE Eee aN eS Tl) Gii…","""Check time, no AM or PM for Co…","""Raven A Wright,10/12/1996,7/1/…","""Raven A Wright""","""Serious Injury,OC Spray Used,B…","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""UO - JDSU (July 2022)_p394""","""Error: less than two boxes fou…","""Cook County Department Of Corr…","""Aqaress: SIUM Y4- SGV MIiVioiV…","""Telephone #: 773.674.3474 O IL…","""07/01/2022""","""SSPE WE Nw EE Pe New ew 1 Giic…","""Check time, no AM or PM for Co…","""Kyle Wunderlich,10/3/1995,4/21…","""Kyle Wunderlich""","""Serious Injury,OC Spray Used,B…","""No injuries""","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""


In [145]:
samples_df.write_excel("7_10_sample.xlsx")